In [1]:
import os, sys
os.chdir("..")
sys.path.append(".")

from src.data_loader import load_matches
matches = load_matches()

1617 380
1718 380
1819 380
1920 380
2021 380
2122 380
2223 380
2324 380
2425 380
2526 380
2627 50


/home/alexrohjak/code/premier-league-predictor/src/data_loader.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches["Season"] = season
/home/alexrohjak/code/premier-league-predictor/src/data_loader.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches["Season"] = season
/home/alexrohjak/code/premier-league-predictor/src/data_loader.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at

In [2]:
matches["FTR"].value_counts(normalize=True)

FTR
H    0.445195
A    0.321299
D    0.233506
Name: proportion, dtype: float64

In [3]:
import pandas as pd
matches["Date"] = pd.to_datetime(matches["Date"], dayfirst = True, format = "mixed")

home = pd.DataFrame({
      "Date": matches["Date"],
      "Team": matches["HomeTeam"],
      "Home": True,
      "Points": matches["FTR"].map({"H": 3, "D": 1, "A": 0}),
  })  

away = pd.DataFrame({
      "Date": matches["Date"],
      "Team": matches["AwayTeam"],
      "Home": False,
      "Points": matches["FTR"].map({"H": 0, "D": 1, "A": 3}),
  })  

team_matches = pd.concat([home, away]).sort_values("Date")
team_matches[team_matches["Team"] == "Liverpool"].tail(5)

,Date,Team,Home,Points
3808,2026-08-23,Liverpool,False,1
3811,2026-08-29,Liverpool,True,1
3820,2026-09-04,Liverpool,False,3
3834,2026-09-12,Liverpool,True,1
3846,2026-09-20,Liverpool,False,3


In [4]:
team_matches["Form"] = (
    team_matches.groupby("Team")["Points"]
    .transform(lambda s: s.shift(1).rolling(5, min_periods = 1).mean())
)
team_matches[team_matches["Team"] == "Tottenham"].tail(6)

,Date,Team,Home,Points,Form
3798,2026-05-24,Tottenham,True,3,1.6
3805,2026-08-22,Tottenham,False,0,2.0
3814,2026-08-29,Tottenham,True,0,1.4
3826,2026-09-05,Tottenham,False,1,0.8
3835,2026-09-12,Tottenham,True,1,0.8
3841,2026-09-19,Tottenham,True,0,1.0


In [5]:
matches["HomeForm"] = team_matches[team_matches["Home"]]["Form"]
matches["AwayForm"] = team_matches[~team_matches["Home"]]["Form"]

/tmp/ipykernel_54576/3640862748.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches["HomeForm"] = team_matches[team_matches["Home"]]["Form"]
/tmp/ipykernel_54576/3640862748.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches["AwayForm"] = team_matches[~team_matches["Home"]]["Form"]


In [6]:
matches[matches["Date"] == "2026-09-19"][["HomeTeam", "AwayTeam", "FTR", "HomeForm", "AwayForm"]]

,HomeTeam,AwayTeam,FTR,HomeForm,AwayForm
3841,Tottenham,Aston Villa,A,1.0,0.8
3842,Brighton,Arsenal,H,1.4,3.0
3843,Everton,Ipswich,H,1.2,1.2
3844,Newcastle,Hull,H,1.0,1.6
3845,Nott'm Forest,Coventry,A,1.2,0.0


In [7]:
matches[["HomeForm", "AwayForm"]].isna().sum()

HomeForm    16
AwayForm    19
dtype: int64

In [8]:
missing = matches[matches["HomeForm"].isna() | matches["AwayForm"].isna()]
missing[["Date", "HomeTeam", "AwayTeam", "HomeForm", "AwayForm"]]

,Date,HomeTeam,AwayTeam,HomeForm,AwayForm
0,2016-08-13,Burnley,Swansea,NaN,NaN
1,2016-08-13,Crystal Palace,West Brom,NaN,NaN
2,2016-08-13,Everton,Tottenham,NaN,NaN
3,2016-08-13,Hull,Leicester,NaN,NaN
4,2016-08-13,Man City,Sunderland,NaN,NaN
5,2016-08-13,Middlesbrough,Stoke,NaN,NaN
6,2016-08-13,Southampton,Watford,NaN,NaN
7,2016-08-14,Arsenal,Liverpool,NaN,NaN
8,2016-08-14,Bournemouth,Man United,NaN,NaN
9,2016-08-15,Chelsea,West Ham,NaN,NaN


In [ ]:
both_missing = matches["HomeForm"].isna() & matches["AwayForm"].isna()
matches = matches[~both_missing]

matches["HomeForm"] = matches["HomeForm"].fillna(0.79)
matches["AwayForm"] = matches["AwayForm"].fillna(0.79)

HomeForm    0
AwayForm    0
dtype: int64

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

features = ["HomeForm", "AwayForm"]

train = matches[matches["Season"] <= "2425"]
test  = matches[matches["Season"] == "2526"]
  
model = LogisticRegression()
model.fit(train[features], train["FTR"])

predictions = model.predict(test[features])
print("Train:", len(train), "Test:", len(test))
print("Accuracy:", accuracy_score(test["FTR"], predictions))

pd.Series(predictions).value_counts()


Train: 3410 Test: 380
Accuracy: 0.4342105263157895


H    258
A    122
Name: count, dtype: int64

In [12]:
test["FTR"].value_counts(normalize=True)

FTR
H    0.426316
A    0.300000
D    0.273684
Name: proportion, dtype: float64